In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests

# -----------------------------
# Config
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
vocab_size = 256          # byte-level
d_model = 128
n_heads = 2
n_layers = 2
block_size = 128          # context length
num_experts = 4
expert_hidden = 256
top_k = 2

batch_size = 32
lr = 3e-4
num_steps = 100000
print_every = 100

In [ ]:
# -----------------------------
# Download Gutenberg Shakespeare
# -----------------------------
url = "https://www.gutenberg.org/files/100/100-0.txt"
print("Downloading Shakespeare from Gutenberg...")
response = requests.get(url)
response.raise_for_status()
corpus = response.text

# Convert to bytes → tensor
data_bytes = corpus.encode("utf-8")
data = torch.tensor(list(data_bytes), dtype=torch.long)

print(f"Dataset loaded: {len(data)} bytes")

Dataset loaded: 5422721 bytes


In [ ]:
# -----------------------------
# Byte-level batch sampler
# -----------------------------
def get_batch():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
# -----------------------------
# Model components
# -----------------------------
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = F.softmax(att, dim=-1)
        y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)

class TinyExpert(nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class MoE(nn.Module):
    def __init__(self, d_model, d_hidden, num_experts, k=2):
        super().__init__()
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList(
            [TinyExpert(d_model, d_hidden) for _ in range(num_experts)]
        )
        self.k = k
        self.num_experts = num_experts

    def forward(self, x):
        B, T, C = x.shape
        scores = self.router(x)
        topk = scores.topk(self.k, dim=-1)
        expert_ids = topk.indices
        expert_weights = F.softmax(topk.values, dim=-1)

        out = torch.zeros_like(x)

        for i in range(self.k):
            eid = expert_ids[..., i]
            w = expert_weights[..., i].unsqueeze(-1)

            expert_out = torch.zeros_like(x)
            for ex_idx in range(self.num_experts):
                mask = (eid == ex_idx)
                if mask.any():
                    x_ex = x[mask]
                    y_ex = self.experts[ex_idx](x_ex)
                    expert_out[mask] = y_ex

            out += w * expert_out

        return out

    def load_balance_loss(self, x):
        with torch.no_grad():
            scores = self.router(x)
            probs = F.softmax(scores, dim=-1)
            p = probs.mean(dim=(0, 1))
        entropy = -(p * (p + 1e-9).log()).sum()
        return -entropy

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, num_experts, expert_hidden, top_k):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.moe = MoE(d_model, expert_hidden, num_experts, k=top_k)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.moe(self.ln2(x))
        return x

class TinyMoELLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers,
                 block_size, num_experts, expert_hidden, top_k):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, num_experts, expert_hidden, top_k)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)

        x = self.token_emb(idx) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, -1),
                targets.view(B * T)
            )
        return logits, loss

    def moe_balance_loss(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.blocks[0].ln1(x)
        return self.blocks[0].moe.load_balance_loss(x)

In [ ]:
# -----------------------------
# Instantiate model
# -----------------------------
model = TinyMoELLM(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    block_size=block_size,
    num_experts=num_experts,
    expert_hidden=expert_hidden,
    top_k=top_k,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [7]:
# -----------------------------
# Training loop
# -----------------------------
model.train()
for step in range(1, num_steps + 1):
    x, y = get_batch()

    logits, ce_loss = model(x, y)
    balance_loss = model.moe_balance_loss(x)

    loss = ce_loss + 0.01 * balance_loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % print_every == 0:
        print(f"step {step:4d} | loss {loss.item():.4f} | ce {ce_loss.item():.4f} | balance {balance_loss.item():.4f}")

step  100 | loss 2.8695 | ce 2.8832 | balance -1.3719
step  200 | loss 2.5848 | ce 2.5984 | balance -1.3625
step  300 | loss 2.4826 | ce 2.4961 | balance -1.3516
step  400 | loss 2.5073 | ce 2.5208 | balance -1.3488
step  500 | loss 2.4851 | ce 2.4986 | balance -1.3464
step  600 | loss 2.4289 | ce 2.4423 | balance -1.3421
step  700 | loss 0.9748 | ce 0.9884 | balance -1.3515
step  800 | loss 0.0923 | ce 0.1058 | balance -1.3504
step  900 | loss 0.0353 | ce 0.0488 | balance -1.3503
step 1000 | loss 0.0292 | ce 0.0428 | balance -1.3528
step 1100 | loss 0.0181 | ce 0.0316 | balance -1.3532
step 1200 | loss 0.0182 | ce 0.0317 | balance -1.3530
step 1300 | loss 0.0134 | ce 0.0269 | balance -1.3516
step 1400 | loss 0.0149 | ce 0.0285 | balance -1.3544
step 1500 | loss 0.0102 | ce 0.0237 | balance -1.3527
step 1600 | loss 0.0087 | ce 0.0222 | balance -1.3523
step 1700 | loss 0.0080 | ce 0.0215 | balance -1.3540
step 1800 | loss 0.0107 | ce 0.0243 | balance -1.3543
step 1900 | loss 0.0133 | ce

In [8]:
# -----------------------------
# Generation demo
# -----------------------------
model.eval()

def generate(model, start_text, max_new_tokens=200):
    with torch.no_grad():
        context = torch.tensor(list(start_text.encode("utf-8")), dtype=torch.long)[None, :].to(device)
        while context.shape[1] < max_new_tokens and context.shape[1] < block_size:
            logits, _ = model(context)
            next_token = torch.distributions.Categorical(logits=logits[:, -1, :]).sample()
            context = torch.cat([context, next_token.unsqueeze(0)], dim=1)
        return bytes(context[0].tolist()).decode("utf-8", errors="ignore")

seed = "To be, or not to be"
print("\n=== SAMPLE ===")
print(generate(model, seed))


=== SAMPLE ===
To be, or not to betttt t nnty tttt tt ttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttt
